In [15]:
import json
import logging
import os
import shutil

In [4]:
logging.basicConfig(
    filename="./logs/DonutDatasetReconstruct.log",
    encoding="utf-8",
    format="%(asctime)s - %(levelname)s - %(message)s",
    level=logging.INFO,
)

In [ ]:
import json

def safe_cast(value):
    try:
        return round(float(value), 2)
    except (ValueError, TypeError):
        return value  # keep string or give null

def extract_label_from_chart(raw_sample):
    text_map = {t["id"]: t["text"] for t in raw_sample.get("text", [])}
    axes = raw_sample.get("axes", {})


    title = next((t["text"] for t in raw_sample["text"] if t["role"] == "chart_title"), None)

    x_axis_title = next((t["text"] for t in raw_sample["text"] if t["role"] == "axis_title" and t["polygon"]["y0"] > 200), None)
    y_axis_title = next((t["text"] for t in raw_sample["text"] if t["role"] == "axis_title" and t["polygon"]["x0"] < 50), None)


    x_tick_ids = [tick["id"] for tick in axes.get("x-axis", {}).get("ticks", [])]
    x_ticks = [text_map.get(tid) for tid in x_tick_ids if tid in text_map]

    y_tick_ids = [tick["id"] for tick in axes.get("y-axis", {}).get("ticks", [])]
    y_ticks = [text_map.get(tid) for tid in y_tick_ids if tid in text_map]

    data_series = [
        {"x": safe_cast(p.get("x")), "y": safe_cast(p.get("y"))}
        for p in raw_sample.get("data-series", [])
        if "x" in p and "y" in p
    ]

    label = {
        "chart-type": raw_sample.get("chart-type", None),
        "title": title,
        "x_axis_title": x_axis_title,
        "y_axis_title": y_axis_title,
        "x_ticks": x_ticks if x_ticks else [],
        "y_ticks": y_ticks if y_ticks else [],
        "data_series": data_series if data_series else []
    }

    return json.dumps(label, ensure_ascii=False, indent=2)

In [ ]:

with open("/Users/yiding/personal_projects/ML/github_repo/donut/data/image_resize/annotations/227c234f207b.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)


label_text = extract_label_from_chart(raw_data)


print(label_text)

{
  "chart-type": "vertical_bar",
  "title": "Estimates, 1950 - 2020: Total population by broad age group, both sexes combined (thousands) - Population aged 15-64 for the year 1975",
  "x_axis_title": "Country",
  "y_axis_title": "Population",
  "x_ticks": [
    "South Africa",
    "South America",
    "South Eastern...",
    "South Korea",
    "South Sudan",
    "Southern Africa",
    "Southern Asia",
    "Southern Europe",
    "Spain",
    "Sri Lanka"
  ],
  "y_ticks": [
    "10000000",
    "9000000",
    "8000000",
    "7000000",
    "6000000",
    "5000000",
    "4000000",
    "3000000",
    "2000000",
    "1000000",
    "0"
  ],
  "data_series": [
    {
      "x": "South Africa",
      "y": 15671980.99
    },
    {
      "x": "South America",
      "y": 16130006.29
    },
    {
      "x": "South Eastern...",
      "y": 14755930.41
    },
    {
      "x": "South Korea",
      "y": 15061280.61
    },
    {
      "x": "South Sudan",
      "y": 13839879.83
    },
    {
      "x": "Sou

In [13]:
def annotation_reconstruct(raw_path_prefix:str,file_txt_path:str,dst_folder:str):
    """
    reconstruct annotation content
    """

    os.makedirs(dst_folder, exist_ok=True)

    with open(file_txt_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    for json_name in lines:
        src_img_path = os.path.join(raw_path_prefix, json_name)
        dst_img_path = os.path.join(dst_folder, json_name)

        with open(src_img_path, "r", encoding="utf-8") as f:
            raw_data = json.load(f)  
        
        label_text = extract_label_from_chart(raw_data)

        # 转成 dict（可选，确保格式化输出）
        label_dict = json.loads(label_text)

        try:
            with open(dst_img_path, "w", encoding="utf-8") as f:
                json.dump(label_dict, f, ensure_ascii=False, indent=2)

        except:
            logging.warning(f"Missing annotations: {dst_img_path}")

    return

In [16]:
vertical_bar_annotation_path_prefix = "/Users/yiding/personal_projects/ML/github_repo/donut/data/image_resize/annotations/"

vertical_bar_dst_folder = "/Users/yiding/personal_projects/ML/github_repo/donut/data/image_classes/vertical_bar_annotations"


annotation_reconstruct(
    raw_path_prefix=vertical_bar_annotation_path_prefix,
    file_txt_path="/Users/yiding/personal_projects/ML/github_repo/donut/data/image_classes/vertical_bar.txt",
    dst_folder=vertical_bar_dst_folder,
)